# B2-020-language-transformers — Practice p12 — Solution

**Type:** constrained-coding · **Difficulty:** intro · **Concepts:** nlp-fine-tuning-protocol

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260812`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb).

## Solution

Freeze the existing encoder objects in place, seed and create one fresh head, and report only its trainable names.

In [ ]:
import torch
from torch import nn

def attach_classification_head(encoder, num_classes=2):
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    torch.manual_seed(20260812)
    head = nn.Linear(8, num_classes)
    trainable_names = [f"head.{name}" for name, parameter in head.named_parameters() if parameter.requires_grad]
    return encoder, head, trainable_names

class ProbeEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(12, 8)
    def forward(self, ids):
        return self.embedding(ids).mean(dim=1)

encoder = ProbeEncoder()
encoder_parameter_ids = [id(parameter) for parameter in encoder.parameters()]
returned_encoder, head, trainable_names = attach_classification_head(encoder)
ids = torch.tensor([[2,4,6], [2,5,7], [2,4,7]], dtype=torch.int64)
logits = head(returned_encoder(ids))

### Answer check

In [ ]:
assert returned_encoder is encoder
assert [id(parameter) for parameter in encoder.parameters()] == encoder_parameter_ids
assert all(not parameter.requires_grad for parameter in encoder.parameters())
assert all(parameter.requires_grad for parameter in head.parameters())
assert trainable_names == ["head.weight", "head.bias"]
assert logits.shape == (3, 2) and logits.dtype == torch.float32